In [ ]:
# ==========================================
# 1. 환경 설정 및 라이브러리
# ==========================================
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm
import gc
import torch

print(f"GPU 활성화 상태: {torch.cuda.is_available()}")

# ==========================================
# 2. 데이터 로드 및 '글자 단위' 전처리
# ==========================================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv('sample_submission.csv')

# [핵심 전략] 길이가 같은 데이터만 사용하여 '밀림 현상' 원천 차단 (97% 데이터 사용)
# 노이즈가 심해 길이가 달라진 3% 데이터는 학습에서 제외하여 모델 혼란을 방지합니다.
train['len_in'] = train['input'].apply(len)
train['len_out'] = train['output'].apply(len)
train_filtered = train[train['len_in'] == train['len_out']].copy()

print(f"전체 데이터: {len(train)} -> 정제된 데이터(길이 일치): {len(train_filtered)}")
print("자모 분리 없이 '글자(Syllable)' 단위로 학습합니다.")

# ==========================================
# 3. Vocab 구축 (글자 단위)
# ==========================================
# 모든 가능한 글자(Character) 수집
all_chars = set()
for text in train['input']: all_chars.update(list(text))
for text in train['output']: all_chars.update(list(text))
for text in test['input']: all_chars.update(list(text))

# 글자 -> ID 매핑
chars = sorted(list(all_chars))
char_to_id = {c: i+1 for i, c in enumerate(chars)}
char_to_id['<PAD>'] = 0
id_to_char = {v: k for k, v in char_to_id.items()}

PAD_ID = char_to_id['<PAD>']
VOCAB_SIZE = len(char_to_id)
print(f"Vocab Size(글자 수): {VOCAB_SIZE} (XGBoost 클래스 개수)")

# ==========================================
# 4. Feature Engineering (Window based)
# ==========================================
WINDOW_SIZE = 7 # 글자 단위이므로 7이면 충분 (좌3, 본인, 우3)
HALF_WINDOW = WINDOW_SIZE // 2

def make_char_dataset(text_series, label_series=None, is_train=True):
    X = []
    y = []

    for idx, text in enumerate(tqdm(text_series, desc="Dataset Making")):
        # 글자 -> ID 변환
        ids = [char_to_id.get(c, 0) for c in list(text)]
        padded_ids = [PAD_ID] * HALF_WINDOW + ids + [PAD_ID] * HALF_WINDOW

        # 정답 ID 변환
        if is_train:
            target_text = label_series.iloc[idx]
            target_ids = [char_to_id.get(c, 0) for c in list(target_text)]
            # 길이가 같은 데이터만 필터링했으므로 len(ids) == len(target_ids) 보장
            loop_len = len(ids)
        else:
            loop_len = len(ids)

        for i in range(loop_len):
            window = padded_ids[i : i + WINDOW_SIZE]
            X.append(window)

            if is_train:
                y.append(target_ids[i])

    # 메모리 최적화를 위해 uint16 사용 (Vocab이 2500개 정도이므로 uint8 불가)
    return np.array(X, dtype=np.uint16), np.array(y, dtype=np.uint16) if is_train else None

print("학습 데이터셋 생성 중...")
X_train_all, y_train_all = make_char_dataset(
    train_filtered['input'],
    train_filtered['output'],
    is_train=True
)

# 검증셋 분리
X_train, X_val, y_train, y_val = train_test_split(X_train_all, y_train_all, test_size=0.1, random_state=42)

# 메모리 정리
del train_filtered, X_train_all, y_train_all
gc.collect()

# ==========================================
# 5. XGBoost 학습 (Native API)
# ==========================================
# Label Encoding (0 ~ N-1)
le = LabelEncoder()
y_combined = np.concatenate([y_train, y_val])
le.fit(y_combined)

y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
num_classes = len(le.classes_)

print(f"실제 학습 클래스 수: {num_classes}")

# DMatrix 변환
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
dval = xgb.DMatrix(X_val, label=y_val_enc)

del X_train, y_train, y_train_enc, y_val_enc
gc.collect()

params = {
    'objective': 'multi:softmax', # 확률 대신 바로 클래스 예측 (메모리 절약 + 속도)
    'num_class': num_classes,
    'learning_rate': 0.1,         # 학습률 약간 상향
    'max_depth': 12,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'device': 'cuda',             # GPU 사용
    'eval_metric': 'merror',      # Multi-class Error Rate
    'seed': 42
}

print("🚀 XGBoost 글자 단위 학습 시작...")
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1500,
    evals=[(dval, 'eval')],
    early_stopping_rounds=30,
    verbose_eval=100
)

# ==========================================
# 6. 추론 및 결과 저장
# ==========================================
print("Test 데이터 변환 중...")
X_test, _ = make_char_dataset(test['input'], is_train=False)

print("추론 진행 중...")
# DMatrix로 변환하여 배치 추론
dtest = xgb.DMatrix(X_test)
pred_enc_ids = model.predict(dtest) # softmax 모드라 바로 클래스 ID 나옴

print("복원 작업 중...")
pred_origin_ids = le.inverse_transform(pred_enc_ids.astype(int))
pred_chars = [id_to_char.get(pid, '') for pid in pred_origin_ids]

# 문장으로 다시 합치기
final_outputs = []
current_idx = 0

for i in tqdm(range(len(test))):
    length = len(test.loc[i, 'input'])
    # 해당 문장의 예측 글자들 가져오기
    sentence_chars = pred_chars[current_idx : current_idx + length]
    current_idx += length

    # 리스트 -> 문자열
    final_outputs.append("".join(sentence_chars))

sample_submission['output'] = final_outputs
sample_submission.to_csv('submission_xgb_syllable.csv', index=False)
print("🎉 완료! submission_xgb_syllable.csv 생성됨.")